# Lab 45: The Invisible Influence

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-45.ipynb)

**What you will do:** test whether a subliminal word prime shifts your friends' brand choices above or below chance, pool the class's results, and see a pretrained image classifier get fooled by a change to a photograph that is too small for a human eye to notice.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Use neutral or invented logos, not real commercial brands — nudging preference between two
actual companies, even informally, is not something to test on people or to publish. Tell your
friend afterwards exactly what you did and why: they could not have consented to a
manipulation they never consciously saw, so debriefing them is part of running this Lab
properly, not an optional extra. If your friend has photosensitive epilepsy or a history of
seizures triggered by flickering images, skip the rapid single-frame video version and use the
slower image-slideshow substitute described in Step 3 instead, or skip this Lab.

## The experiment in the book

In Lab 45 you built a short video clip that flashed a single word — positive ("wonderful") or
negative ("danger") — for about 30–60 milliseconds before showing a brand logo, then asked a
friend which of two brands they preferred. Subliminal positive primes tend to increase
preference for the following brand; negative primes decrease it. The effect is small but
statistically reliable across many participants, and almost everyone denies noticing anything
before the logo.

## Record your results

Test as many friends as you can, alternating which prime valence each one sees. For each,
record which prime they saw and whether they chose the brand that was paired with it.

In [ ]:
# example data: replace with your own trials
trial_data = pd.DataFrame({
    "participant": ["Friend A", "Friend B", "Friend C", "Friend D", "Friend E", "Friend F"],
    "prime": ["positive", "positive", "positive", "negative", "negative", "negative"],
    "chose_primed_brand": [True, True, False, False, False, True],  # example data
})
display(trial_data)

rate_by_prime = trial_data.groupby("prime")["chose_primed_brand"].mean()
display(rate_by_prime.to_frame("proportion choosing primed brand"))
rate_by_prime.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.axhline(0.5, color="grey", linestyle="--", label="chance")
plt.ylim(0, 1); plt.legend(); plt.ylabel("proportion choosing the primed brand"); plt.show()

## Compare

The text describes the effect of subliminal priming on preference as "small but
statistically reliable across many participants" — it gives no specific percentage. The
natural benchmark, absent any influence, is chance: 50%, since both brands were equally
unfamiliar and equally likely to be chosen.

In [ ]:
positive_rate = rate_by_prime.get("positive", float("nan"))
negative_rate = rate_by_prime.get("negative", float("nan"))
print(f"Positive prime -> chose the primed brand: {positive_rate:.0%}")
print(f"Negative prime -> chose the primed brand: {negative_rate:.0%}")
print("(For the negative-prime group, choosing the primed brand despite the negative word should")
print("be LESS common than chance, not more, if the priming account is correct.)")
if positive_rate > 0.5 and negative_rate < 0.5:
    print("Both directions match the account: positive priming nudges choice up, negative priming nudges it down.")
else:
    print("With a handful of friends, an effect the text itself calls 'small' can easily wash out or reverse;")
    print("this is exactly why such effects are normally reported across many participants, not a handful.")

## The AI side

The book compares subliminal priming to **adversarial examples**: a change to an input, too
small for a person to notice, that is enough to flip an AI system's output (Goodfellow,
Shlens, & Szegedy, 2015, *ICLR*). Below, a pretrained image classifier is shown a photograph
it classifies correctly, then the same photograph with a tiny perturbation computed by the
**Fast Gradient Sign Method** — nudging every pixel slightly in the direction that most
increases the model's error. Watch whether a change too small to see with the naked eye is
enough to flip the model's answer.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()
labels = models.ResNet18_Weights.IMAGENET1K_V1.meta["categories"]

try:
    url = "https://pytorch.org/tutorials/_static/img/panda.jpg"
    with urllib.request.urlopen(url, timeout=10) as response:
        image = Image.open(io.BytesIO(response.read())).convert("RGB")
except Exception:
    image = Image.new("RGB", (224, 224), (120, 150, 90))  # fallback if the download fails

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x = preprocess(image).unsqueeze(0)
x.requires_grad = True

output = model(x)
original_pred = output.argmax(dim=1).item()
loss = F.cross_entropy(output, torch.tensor([original_pred]))
model.zero_grad()
loss.backward()

epsilon = 0.03  # a tiny, almost invisible perturbation -- try raising this if nothing flips
perturbed_x = (x + epsilon * x.grad.sign()).detach()

with torch.no_grad():
    perturbed_output = model(perturbed_x)
perturbed_pred = perturbed_output.argmax(dim=1).item()

print(f"Original prediction:  {labels[original_pred]}")
print(f"Perturbed prediction: {labels[perturbed_pred]}")
print(f"Largest single-pixel change (normalised units): {(perturbed_x - x).abs().max().item():.4f}")
print("Flipped!" if perturbed_pred != original_pred else "Unchanged at this epsilon -- try a larger epsilon above.")

If the perturbed prediction differs from the original, the model has been fooled by a change
too small to see — exactly the way a 30-millisecond word can shift a choice without being
consciously seen. If nothing flipped, try raising `epsilon`; a larger perturbation is more
likely to fool the network, at the cost of eventually becoming visible, just as a subliminal
prime becomes a supraliminal (consciously visible) one if it is shown for too long or too
intensely. The mechanisms are otherwise unrelated: the adversarial perturbation is computed
deliberately, by following the gradient that most increases the model's error on this
specific image; the subliminal word activates a pre-existing associative network. There is no
brain equivalent of computing an optimal "attack" tailored to one person's preferences.

## Pool the class data

Pool the class: each pair adds one row (an anonymous participant ID, which prime valence
they saw, and whether they chose the primed brand). The example rows below are placeholders
so the cell runs.

In [ ]:
import io
csv_text = """participant_id,prime,chose_primed_brand
P01,positive,True
P02,positive,True
P03,positive,False
P04,negative,False
P05,negative,True
P06,negative,False
P07,positive,True
P08,negative,False
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
rate_by_prime = class_df.groupby("prime")["chose_primed_brand"].mean()
display(rate_by_prime.to_frame("proportion choosing primed brand"))

for prime, group in class_df.groupby("prime"):
    boots = [group.sample(len(group), replace=True)["chose_primed_brand"].mean() for _ in range(5000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{prime}: {rate_by_prime[prime]:.0%} (95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(group)})")

rate_by_prime.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.axhline(0.5, color="grey", linestyle="--", label="chance")
plt.ylim(0, 1); plt.legend(); plt.ylabel("proportion choosing the primed brand"); plt.show()

## Questions to think about

1. Your negative-prime group's "chose the primed brand" rate should, if the effect is real, fall BELOW 50%, while your positive-prime group's rate should rise ABOVE 50%. Did both hold in your data? If only one did, what would you want to check before concluding the effect works in only one direction?
2. The text distinguishes small, short-lived semantic/affective priming (well replicated) from larger claims about behavioural priming (largely not replicated, e.g. Doyen et al., 2012). Which kind is the brand-choice task in this Lab closer to, and does that change how much weight you put on a single small-sample result?
3. In the ai_lab demo, what do you expect to happen to the model's prediction as `epsilon` grows larger? At what point does an adversarial perturbation stop being the AI equivalent of "subliminal" and start being more like a supraliminal prime?
4. The text argues that the deeper modern worry is not perceptual subliminal influence but "supraliminal influence that operates below the threshold of attention" -- background music, ballot order, and so on. Sketch one record-style measurement you could add to this notebook to test an attention-level influence rather than a perception-level one.

## Challenge

Design your own subliminal-prime pair using two invented, non-commercial logos and two
neutral or invented words instead of loaded ones, to check whether even mild valence
differences shift choice. Disclose the manipulation to your friend immediately afterwards.

In [ ]:
# Example data: replace with your own invented-logo trial.
my_positive_word = "sunshine"              # example: an invented/neutral positive word
my_negative_word = "puddle"                # example: an invented/neutral negative word
friend_chose_positive_primed_brand = True  # example data

print(f"Positive prime used: '{my_positive_word}' | Negative prime used: '{my_negative_word}'")
print(f"Chose the brand paired with the positive prime: {friend_chose_positive_primed_brand}")
print("Remember to tell your friend afterwards exactly what you did -- they could not have consented")
print("to something they never consciously saw. The values above are example data.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-45.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")